# 📬 Интеллектуальный помощник по дайджесту электронной почты (EmailDigestAgent)

## Интеллектуальная классификация почты и генерация сводок и ежедневных отчётов на платформе HelloAgents.

**Основной процесс:** получение писем (IMAP) → интеллектуальная классификация → сводка ИИ → ежедневный отчёт.

**Поддерживаемые почтовые сервисы:** QQ / 163 / 126 / Gmail / Outlook и любой ящик с IMAP.



## Часть 1. Настройка среды
### 1.1 Установка зависимостей


In [ ]:
# Установка зависимостей (можно пропустить, если уже установлено)
# hello-agents + базовые инструменты; imaplib — стандартная библиотека Python
# !pip install -q hello-agents python-dotenv rich




### 1.2 Импортируйте библиотеку и настройте LLM


In [ ]:
import os
import json
import re
import email
import imaplib
from email.header import decode_header
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional
from dataclasses import dataclass, field
from collections import Counter

# HelloAgents
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool

# Utils
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()
load_dotenv(override=True)

# Init LLM
llm = HelloAgentsLLM()
model_id = os.getenv("LLM_MODEL_ID", "gpt-4o-mini")
print("✅ Окружение настроено")
print(f"   Модель LLM: {model_id}")



## Часть 2. Моделирование данных электронной почты
Встроенные 10 смоделированных электронных писем, охватывающих различные сценарии для демонстрационного режима.


In [ ]:
# ========================================
# Набор демо-писем (10 штук, разные сценарии)
# ========================================

DEMO_EMAILS = [
    {
        "id": "1",
        "from": "boss@company.com",
        "subject": "Срочно: подтвердите бюджет Q3 до конца рабочего дня",
        "date": "2026-07-02 08:30:00",
        "body": "Уважаемые руководители отделов, во вложении черновик бюджета Q3. Просьба проверить и подтвердить до конца рабочего дня. Замечания — на завтрашнем бюджетном совещании. Отделы без подтверждения будут учтены по текущему варианту.\n\nВложение: Q3_Budget_v2.xlsx"
    },
    {
        "id": "2",
        "from": "client@vip-corp.com",
        "subject": "Re: Замечания к пунктам 5 и 8 договора",
        "date": "2026-07-02 09:15:00",
        "body": "Здравствуйте, юридический отдел проверил проект договора и высказал серьёзные возражения по пункту 5 (оплата) и пункту 8 (ответственность). Особенно нужно скорректировать размер неустойки. Просим организовать онлайн-встречу — хотим решить вопрос на этой неделе."
    },
    {
        "id": "3",
        "from": "hr@company.com",
        "subject": "【Уведомление】Система годовой оценки закрывается в пятницу",
        "date": "2026-07-02 10:00:00",
        "body": "Коллеги, самооценка и взаимная оценка закроются в пятницу (4 июля) в 17:00. В системе у вас остались 2 невыполненные взаимные оценки. Пожалуйста, завершите оценку вовремя — просрочка повлияет на итоги года.\n\nСсылка: http://hr.company.com/performance"
    },
    {
        "id": "4",
        "from": "github-notifications@github.com",
        "subject": "[hello-agents] New PR #156: Add EmailDigestAgent project",
        "date": "2026-07-02 10:30:00",
        "body": "Pull Request #156 opened by WHS.\n\nTitle: Add EmailDigestAgent project\nBranch: feature/email-digest to main\nFiles changed: 5\n\nView on GitHub: https://github.com/datawhalechina/hello-agents/pull/156"
    },
    {
        "id": "5",
        "from": "newsletter@techweekly.com",
        "subject": "Tech Weekly #234: AI Agent trends and tools roundup",
        "date": "2026-07-02 11:00:00",
        "body": "Подборка недели:\n1. OpenAI представила новое поколение Agent-фреймворка\n2. Крупное обновление LangChain v0.3\n3. Разбор лучших практик мультиагентного взаимодействия\n4. Open source: 5 проектов Agent, за которыми стоит следить\n\nЧитать: https://techweekly.com/234"
    },
    {
        "id": "6",
        "from": "li.team@company.com",
        "subject": "Обновление повестки проектного митинга в 15:00",
        "date": "2026-07-02 12:00:00",
        "body": "Hi team, обновлена повестка сегодняшнего митинга в 15:00:\n\n1. Sprint review (15 мин)\n2. Прогресс по исправлению багов (20 мин)\n3. Ревью новых требований — модуль прав пользователей (25 мин)\n\nПожалуйста, подготовьте обновления по Jira board."
    },
    {
        "id": "7",
        "from": "marketing@online-shop.com",
        "subject": "Последний день распродажи! Скидки от 50%",
        "date": "2026-07-02 13:00:00",
        "body": "Обратный отсчёт распродажи! Последние 24 часа!\n\nСкидки от 50% на весь ассортимент\nБесплатная доставка от 299\nПодарок при заказе от 599\n\nКупить: https://shop.example.com/sale\nОтписаться: ответьте TD"
    },
    {
        "id": "8",
        "from": "noreply@calendar.google.com",
        "subject": "Напоминание: квартальный обзор завтра в 10:00",
        "date": "2026-07-02 14:00:00",
        "body": "Напоминание календаря:\n\nСобытие: квартальный обзор\nВремя: 3 июля 2026, 10:00–12:00\nМесто: переговорная A, 3 этаж / Zoom\nУчастники: все руководители отделов\n\nПодготовьте материалы для доклада."
    },
    {
        "id": "9",
        "from": "spam-bot@random-mail.xyz",
        "subject": "Congratulations! You've won a FREE iPhone!",
        "date": "2026-07-02 15:00:00",
        "body": "Dear Winner,\n\nYour email address has been selected in our annual lottery! You have won a brand new iPhone 16 Pro Max!\n\nTo claim your prize, click: http://suspicious-link.xyz/claim\n\nHurry! This offer expires in 24 hours!"
    },
    {
        "id": "10",
        "from": "team-lead@company.com",
        "subject": "Онбординг нового коллеги и назначение mentor",
        "date": "2026-07-02 16:30:00",
        "body": "Добрый день, в понедельник к нам присоединится новый коллега (frontend, 2 года опыта). Нужно организовать:\n\n1. План онбординга (HR уже отправил письмо)\n2. Назначить mentor (желательно опытный frontend-разработчик)\n3. Подготовить окружение и документ для новичков\n\nПожалуйста, до конца дня напишите в чат, кто может быть mentor!"
    }
]

print(f"✅ Загружено {len(DEMO_EMAILS)} демо-писем")

preview = "\n".join([
    f"  {i+1}. [{e['from']}] {e['subject'][:50]}..."
    for i, e in enumerate(DEMO_EMAILS)
])
console.print(Panel.fit(preview, title="📧 Список демо-писем", style="blue"))



## Часть 3: Определение инструмента
Определите два основных инструмента:- **EmailFetchTool**: получайте электронные письма по протоколу IMAP (поддерживает QQ/163/126/Gmail/Outlook) или используйте смоделированные данные.- **EmailDigestTool**: классифицируйте и суммируйте списки адресов электронной почты с помощью LLM.


In [ ]:
# ========================================
# Инструмент 1: получение писем (универсальный IMAP)
# ========================================

def _decode_mime_header(header_value: str) -> str:
    if not header_value:
        return ""
    parts = decode_header(header_value)
    decoded = ""
    for text, charset in parts:
        if isinstance(text, bytes):
            try:
                decoded += text.decode(charset or 'utf-8', errors='replace')
            except (LookupError, UnicodeDecodeError):
                decoded += text.decode('utf-8', errors='replace')
        else:
            decoded += str(text)
    return decoded


def _parse_email_body(msg) -> str:
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            disposition = str(part.get("Content-Disposition", ""))
            if content_type == "text/plain" and "attachment" not in disposition:
                payload = part.get_payload(decode=True)
                if payload:
                    charset = part.get_content_charset() or 'utf-8'
                    try:
                        body = payload.decode(charset, errors='replace')
                    except (LookupError, UnicodeDecodeError):
                        body = payload.decode('utf-8', errors='replace')
                    break
        if not body:
            for part in msg.walk():
                if (part.get_content_type() == "text/html"
                        and "attachment" not in str(part.get("Content-Disposition", ""))):
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or 'utf-8'
                        try:
                            body = payload.decode(charset, errors='replace')
                            body = re.sub(r'<[^>]+>', '', body)
                            body = re.sub(r'\s+', ' ', body).strip()
                        except Exception:
                            body = payload.decode('utf-8', errors='replace')
                    break
    else:
        payload = msg.get_payload(decode=True)
        if payload:
            charset = msg.get_content_charset() or 'utf-8'
            try:
                body = payload.decode(charset, errors='replace')
            except Exception:
                body = payload.decode('utf-8', errors='replace')
    return body[:3000]


class EmailFetchTool(Tool):
    def __init__(self, use_demo: bool = True, demo_emails=None):
        super().__init__(
            name="email_fetch",
            description="Получение непрочитанных писем из входящих"
        )
        self.use_demo = use_demo
        self.demo_emails = demo_emails or DEMO_EMAILS

    def _read_imap_config(self) -> tuple:
        server = os.getenv("IMAP_SERVER") or ""
        port = int(os.getenv("IMAP_PORT") or "0")
        username = os.getenv("IMAP_USERNAME") or ""
        password = os.getenv("IMAP_PASSWORD") or ""
        if not (username and password and server):
            try:
                with open("config/email_config.json", "r", encoding="utf-8") as f:
                    cfg = json.load(f)
                imap_cfg = cfg.get("imap", {})
                server = server or imap_cfg.get("server", "imap.qq.com")
                port = port or imap_cfg.get("port", 993)
                username = username or imap_cfg.get("username", "")
                password = password or imap_cfg.get("password", "")
            except FileNotFoundError:
                pass
        return server or "imap.qq.com", port or 993, username, password

    def _fetch_via_imap(self, max_emails: int = 50, hours: int = 24) -> list:
        server, port, username, password = self._read_imap_config()
        if not username or not password:
            raise ValueError("Почтовый ящик не настроен")

        console.print(f"[dim]  Подключение {server}:{port}  пользователь: {username}...[/dim]")
        mail = imaplib.IMAP4_SSL(server, port)
        mail.login(username, password)

        # Выбор INBOX — попытки: без кавычек, с кавычками, readonly
        select_status = None
        for name, rdonly in [("INBOX", False), ('"INBOX"', False), ("INBOX", True)]:
            select_status, select_data = mail.select(name, readonly=rdonly)
            console.print(f"[dim]  select({name!r}, readonly={rdonly}) -> {select_status!r}[/dim]")
            if select_status == "OK":
                break
        if select_status != "OK":
            raise ValueError(f"Не удалось выбрать INBOX (статус: {select_status!r})")

        since_date = (datetime.now() - timedelta(hours=hours)).strftime("%d-%b-%Y")
        search_criteria = f'(UNSEEN SINCE {since_date})'
        status, message_ids = mail.search(None, search_criteria)
        if status != "OK":
            mail.logout()
            return []

        ids = message_ids[0].split()
        ids = list(reversed(ids))[:max_emails]
        emails = []
        for msg_id in ids:
            try:
                status, msg_data = mail.fetch(msg_id, "(RFC822)")
                if status != "OK":
                    continue
                raw_email = msg_data[0][1]
                msg = email.message_from_bytes(raw_email)
                msg_id_str = msg_id.decode() if isinstance(msg_id, bytes) else msg_id
                emails.append({
                    "id": msg_id_str,
                    "from": _decode_mime_header(msg.get("From", "")),
                    "subject": _decode_mime_header(msg.get("Subject", "(без темы)")),
                    "date": msg.get("Date", ""),
                    "body": _parse_email_body(msg)
                })
            except Exception as e:
                console.print(f"[yellow]  ⚠️ Ошибка разбора письма: {e}[/yellow]")
                continue

        mail.logout()
        console.print(f"[dim]  Соединение закрыто[/dim]")
        return emails

    def run(self, hours: int = 24, max_emails: int = 50) -> str:
        if self.use_demo:
            emails = self.demo_emails[:max_emails]
        else:
            try:
                emails = self._fetch_via_imap(max_emails=max_emails, hours=hours)
            except imaplib.IMAP4.error as e:
                console.print(f"[red]❌ Ошибка подключения IMAP: {e}[/red]")
                raise RuntimeError(f"Ошибка подключения IMAP: {e}") from e
            except ValueError:
                raise
            except Exception as e:
                console.print(f"[red]❌ Неизвестная ошибка: {e}[/red]")
                raise

        if not emails:
            return json.dumps({"message": "Нет новых непрочитанных писем", "emails": []}, ensure_ascii=False, indent=2)
        return json.dumps({"count": len(emails), "fetch_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "emails": emails}, ensure_ascii=False, indent=2)

    def get_parameters(self):
        from hello_agents.tools import ToolParameter
        return []

print("✅ EmailFetchTool defined (IMAP version)")



In [ ]:
# ========================================
# Инструмент 2: сводка и классификация писем (на LLM)
# ========================================

class EmailDigestTool(Tool):
    """Классификация, сводка и приоритет писем с помощью LLM"""

    CATEGORIES = ["работа", "клиент", "личное", "уведомление", "реклама", "спам"]
    PRIORITIES = ["высокий", "средний", "низкий"]

    def __init__(self, llm=None, use_demo: bool = True, demo_emails=None):
        super().__init__(
            name="email_fetch",
            description="Получение непрочитанных писем. Демо-режим и реальный IMAP."
        )
        self.llm = llm
        self.use_demo = use_demo
        self.demo_emails = demo_emails or DEMO_EMAILS

    def _build_prompt(self, emails_json: str, count: int) -> str:
        cats = ", ".join(self.CATEGORIES)
        prios = ", ".join(self.PRIORITIES)
        return f"""Вы — профессиональный помощник по анализу почты. Проанализируйте список писем и верните структурированный результат.

Данные писем (JSON):
{emails_json}

Правила анализа:
1. Категория — выберите одну из [{cats}]
2. Приоритет — выберите из [{prios}]:
   высокий: требует немедленного внимания (руководитель, клиент, договор, дедлайн)
   средний: нужно обработать, но не срочно (проект, коллеги, HR)
   низкий: можно отложить (рассылки, реклама, системные уведомления, спам)
3. Краткая сводка — одно предложение на русском с ключевым действием или сроком
4. Ключевое действие — 1–2 предложения при необходимости ответа; иначе «нет»

Строго верните JSON в формате ниже (только JSON, без лишнего текста):
{{
  "analyzed_at": "текущее время",
  "total": {count},
  "items": [
    {{
      "id": "ID письма",
      "from": "отправитель",
      "subject": "тема",
      "category": "категория",
      "priority": "приоритет",
      "summary": "краткая сводка",
      "action": "ключевое действие"
    }}
  ]
}}"""

    def run(self, emails_json: str) -> str:
        try:
            data = json.loads(emails_json)
            count = data.get("count", len(data.get("emails", [])))
            prompt = self._build_prompt(emails_json, count)
            response = self.llm.invoke([{"role": "user", "content": prompt}])
            result = response.content.strip()
            
            if result.startswith("```"):
                result = re.sub(r'^```(?:json)?\s*', '', result)
                result = re.sub(r'\s*```$', '', result)

            parsed = json.loads(result)
            return json.dumps(parsed, ensure_ascii=False, indent=2)

        except json.JSONDecodeError as e:
            return json.dumps({
                "error": "Некорректный формат ответа LLM",
                "detail": str(e)
            }, ensure_ascii=False, indent=2)
        except Exception as e:
            return json.dumps({
                "error": "Ошибка при анализе",
                "detail": str(e)
            }, ensure_ascii=False, indent=2)
    def get_parameters(self):
        from hello_agents.tools import ToolParameter
        return []


print("✅ EmailDigestTool defined")



## Часть 4. Конвейер создания ежедневных отчетов
Организуйте весь процесс: Получить → Сводка классификации → Статистика → Создание ежедневного отчета → Сохранить файл.


In [ ]:
# ========================================
# Конвейер генерации почтового дайджеста
# ========================================

@dataclass
class EmailDigestPipeline:
    """Конвейер: получение (IMAP/демо) -> LLM -> Markdown-отчёт"""

    llm: any
    use_demo: bool = True
    fetch_tool: EmailFetchTool = field(init=False)
    digest_tool: EmailDigestTool = field(init=False)

    def __post_init__(self):
        self.fetch_tool = EmailFetchTool(use_demo=self.use_demo)
        self.digest_tool = EmailDigestTool(llm=self.llm)

    def run(self, hours: int = 24, max_emails: int = 50) -> str:
        """Выполнить полный конвейер и вернуть Markdown-отчёт"""
        mode = "демо-данные" if self.use_demo else "реальный IMAP"
        console.print(Panel.fit(
            f"Режим: {mode}  |  Период: последние{hours}h  |  Лимит: {max_emails} писем",
            title="📬 EmailDigestAgent", style="cyan"
        ))

        # Step 1: Fetch
        console.print("\n📥 [1/4] Получение писем...", style="cyan")
        raw = self.fetch_tool.run(hours=hours, max_emails=max_emails)
        data = json.loads(raw)
        count = data.get("count", len(data.get("emails", [])))
        if count == 0:
            return self._empty_report()
        console.print(f"   ✅ {count}  писем")

        # Step 2: LLM digest
        console.print("\n🧠 [2/4] LLM-классификация и сводка...", style="cyan")
        digest_raw = self.digest_tool.run(raw)
        digest = json.loads(digest_raw)
        items = digest.get("items", [])
        if "error" in digest:
            console.print(f"   ⚠️  {digest['error']}")
        console.print(f"   ✅ Проанализировано {len(items)}  писем")

        # Step 3: Stats
        console.print("\n📊 [3/4] Статистика...", style="cyan")
        stats = self._compute_stats(items)
        console.print(f"   🔴выс:{stats['priorities'].get('высокий',0)}  🟡ср:{stats['priorities'].get('средний',0)}  🟢низ:{stats['priorities'].get('низкий',0)}")

        # Step 4: Report
        console.print("\n📝 [4/4] Генерация отчёта...", style="cyan")
        report = self._generate_report(stats, items)

        os.makedirs("output", exist_ok=True)
        path = f"output/email_digest_{datetime.now().strftime('%Y%m%d')}.md"
        with open(path, 'w', encoding='utf-8') as f:
            f.write(report)
        console.print(f"\n📄 Сохранено: {path}", style="bold green")
        return report

    def _compute_stats(self, items):
        cats = Counter(i.get("category", "other") for i in items)
        prios = Counter(i.get("priority", "средний") for i in items)
        actions = sum(1 for i in items if i.get("action", "нет") != "нет")
        return {"total": len(items), "categories": dict(cats),
                "priorities": dict(prios), "need_action": actions}

    def _generate_report(self, stats, items):
        now = datetime.now().strftime("%Y-%m-%d %H:%M")
        src = "демо-данные" if self.use_demo else "IMAP-входящие"
        order = {"высокий": 0, "средний": 1, "низкий": 2}
        sorted_items = sorted(items, key=lambda x: order.get(x.get("priority", "средний"), 1))

        lines = [
            f"# 📬 Почтовый дайджест",
            f"**Время генерации：** {now}  |  **Источник：** {src}",
            f"---",
            f"## 📊 Обзор",
            f"| Показатель | Значение |",
            f"|------|------|",
            f"| Всего | {stats['total']} писем |",
            f"| 🔴 Высокий приоритет | {stats['priorities'].get('высокий', 0)} |",
            f"| 🟡 Средний приоритет | {stats['priorities'].get('средний', 0)} |",
            f"| 🟢 Низкий приоритет | {stats['priorities'].get('низкий', 0)} |",
            f"| 📋 Требуют действия | {stats['need_action']} |",
            f"",
            f"### Распределение по категориям",
        ]
        for cat, cnt in stats['categories'].items():
            lines.append(f"- **{cat}**: {cnt} писем {'█' * cnt}")
        lines.append(f"---")

        for pri, label in [("высокий", "🔴 Высокий приоритет — немедленно"),
                            ("средний", "🟡 Средний приоритет — сегодня"),
                            ("низкий", "🟢 Низкий приоритет — можно отложить")]:
            pitems = [i for i in sorted_items if i.get("priority") == pri]
            if not pitems:
                continue
            lines.append(f"## {label}")
            lines.append(f"| # | Отправитель | Тема | Тип | Краткая сводка |")
            lines.append(f"|---|--------|------|------|-----------|")
            for idx, item in enumerate(pitems, 1):
                fr = item.get("from", "")
                subj = item.get("subject", "")
                cat = item.get("category", "-")
                summ = item.get("summary", "-")
                lines.append(f"| {idx} | {fr} | {subj} | {cat} | {summ} |")
            lines.append("")
            if pri in ("высокий", "средний"):
                acts = [i for i in pitems if i.get("action", "нет") != "нет"]
                if acts:
                    lines.append("**📋 Действия：**")
                    for a in acts:
                        lines.append(f"- [{a.get('category','')}] **{a.get('subject','')}** — {a.get('action','')}")
                    lines.append("")

        lines.extend(["---", "*EmailDigestAgent · HelloAgents*"])
        return "\n".join(lines)

    def _empty_report(self):
        now = datetime.now().strftime("%Y-%m-%d %H:%M")
        return (f"# 📬 Почтовый дайджест\n\n**{now}**\n\n## 🎉 Входящие пусты！\n\n"
                f"Непрочитанных писем нет。\n\n---\n*EmailDigestAgent · HelloAgents*")

print("✅ EmailDigestPipeline defined")



## Часть 5: Функциональная демонстрация
### 5.1 Демо-режим: создание ежедневного отчета по электронной почте


In [ ]:
# Создание конвейера (демо-режим) и запуск
pipeline = EmailDigestPipeline(llm=llm, use_demo=True)
report = pipeline.run(hours=24, max_emails=20)

# Отображение отчёта
console.print("\n" + "=" * 60)
console.print("📬 Почтовый дайджест", style="bold cyan")
console.print("=" * 60 + "\n")
console.print(Markdown(report))



### 5.2 Просмотр сохраненных файлов ежедневных отчетов


In [ ]:
import glob
reports = sorted(glob.glob("output/email_digest_*.md"), reverse=True)
if reports:
    print(f"Latest: {reports[0]}")
    with open(reports[0], 'r', encoding='utf-8') as f:
        console.print(Markdown(f.read()))
else:
    print("No reports yet. Run the pipeline first.")


### 5.3 Визуализация статистики данных


In [ ]:
# Статистика по категориям и приоритетам
emails_data = json.loads(pipeline.fetch_tool.run(hours=24, max_emails=20))
digest_data = json.loads(pipeline.digest_tool.run(json.dumps(emails_data)))
items = digest_data.get("items", [])

table = Table(title="📊 Статистика анализа почты")
table.add_column("Показатель", style="cyan")
table.add_column("Значение", style="white")
table.add_column("Пояснение", style="dim")
table.add_row("Всего", str(len(items)), "")
table.add_row("Срочно (высокий)", str(sum(1 for i in items if i.get("priority")=="высокий")), "🔴")
table.add_row("Сегодня (средний)", str(sum(1 for i in items if i.get("priority")=="средний")), "🟡")
table.add_row("Можно отложить (низкий)", str(sum(1 for i in items if i.get("priority")=="низкий")), "🟢")
table.add_row("Требуют действия", str(sum(1 for i in items if i.get("action","нет")!="нет")), "📋")
console.print(table)

console.print("\n📂 Распределение по категориям：")
cats = Counter(i.get("category","other") for i in items)
for cat, cnt in cats.most_common():
    bar = "█" * (cnt * 3)
    console.print(f"  {cat:8s}  {cnt:2d}  {bar}")



## Часть 6. Реальный доступ к электронной почте (общий протокол IMAP)
Поддерживает **почтовый ящик QQ/163/126/Gmail/Outlook** и другие почтовые ящики, поддерживающие IMAP.
### Быстрая проверка конфигурации IMAP для каждого почтового ящика


In [ ]:
print("=" * 60)
print("📧 Руководство по подключению реального ящика")
print("=" * 60)
print()
print("【Настройки IMAP для почтовых сервисов】")
print()
print("  QQ Mail:")
print("    server: imap.qq.com  port: 993")
print("    Шаги: веб-версия QQ Mail -> Настройки -> Аккаунт -> включить IMAP/SMTP")
print("          -> подтвердить безопасность -> получить код авторизации (не пароль QQ)")
print()
print("  NetEase 163:")
print("    server: imap.163.com  port: 993")
print("    Шаги: веб-версия NetEase -> Настройки -> POP3/SMTP/IMAP -> включить IMAP")
print("          -> получить код авторизации")
print()
print("  NetEase 126:")
print("    server: imap.126.com  port: 993")
print("    Шаги: как для 163")
print()
print("  Gmail:")
print("    server: imap.gmail.com  port: 993")
print("    Шаги: аккаунт Google -> Безопасность -> 2FA -> пароль приложения")
print()
print("  Outlook/Hotmail:")
print("    server: outlook.office365.com  port: 993")
print("    Шаги: аккаунт Microsoft -> Безопасность -> пароль приложения")
print()
print("【Два способа настройки】")
print()
print("  Способ 1 (рекомендуется) — файл .env:")
print("    IMAP_SERVER=imap.qq.com")
print("    IMAP_PORT=993")
print("    IMAP_USERNAME=your_email@qq.com")
print("    IMAP_PASSWORD=ваш_код_авторизации")
print()
print("  Способ 2 — config/email_config.json:")
print("    измените server / port / username / password в секции imap")
print()
print("【Запуск реального режима】")
print("  pipeline = EmailDigestPipeline(llm=llm, use_demo=False)")
print("  report = pipeline.run(hours=24, max_emails=50)")
print("  console.print(Markdown(report))")
print()
print("=" * 60)



### Демонстрация реального режима
Перед запуском завершите настройку в соответствии с приведенным выше руководством. При сбое соединения будет напрямую сообщено об ошибке и отображена причина, чтобы облегчить устранение неполадок.


In [ ]:
# ========================================
# Реальный режим IMAP (раскомментируйте)
# ========================================
#
# После настройки IMAP_SERVER/IMAP_PORT/IMAP_USERNAME/IMAP_PASSWORD в .env:
#
# pipeline_real = EmailDigestPipeline(llm=llm, use_demo=False)
# report = pipeline_real.run(hours=24, max_emails=50)
# console.print(Markdown(report))

print("💡 Раскомментируйте код выше и настройте IMAP в .env для запуска реального режима.")
print("При ошибке IMAP будет показана причина для диагностики.")



## Часть 7: Краткое описание проекта

### ✅ Реализованные функции
- **Получение почты**: IMAP (QQ/163/126/Gmail/Outlook) + демо-режим; при сбое в реальном режиме ошибка сообщается напрямую
- **Декодирование MIME**: корректная обработка кириллических тем, имён отправителей, multipart и HTML
- **Классификация LLM**: 6 типов (работа/клиент/личное/уведомление/реклама/спам) + 3 уровня приоритета
- **Сводка ИИ**: семантическое понимание письма, краткая сводка и ключевые действия на русском
- **Ежедневный отчёт**: Markdown-дайджест по приоритетам со статистикой и списком действий
- **Сохранение файлов**: автоматически в каталог output/

### 🎯 Особенности дизайна
- **Реальная боль**: перегруженный почтовый ящик — ежедневная проблема офисных работников
- **Универсальный IMAP**: не привязан к Gmail, корпоративные ящики поддерживаются из коробки
- **Строгие ошибки**: при сбое IMAP в реальном режиме — явное исключение для диагностики
- **Понятный конвейер**: Получить → Классификация/Сводка → Отчёт, четыре шага

### 🔮 Перспективы
- Планировщик (автозапуск каждое утро в 8:00)
- Отправка дайджеста в WeChat/DingTalk
- Недельная/месячная статистика трендов

---

<div align="center">
<strong>📬 EmailDigestAgent — пусть ИИ читает ваши входящие</strong><br>HelloAgents · Совместное создание сообщества Datawhale</div>

